# Архив исходного исследования

Учебный notebook до выделения CLI-версии. Выводы ячеек очищены перед публикацией. Для воспроизводимого запуска используйте README и модуль portfolio. Исторические формулировки и схема эксперимента сохранены; ограничения оценки описаны в README.


# Прогнозирование температуры звезды

## Введение

### Описание  
Обсерватория «Небо на ладони» сталкивается с задачей автоматизированного определения температуры на поверхности обнаруженных звёзд. Традиционные методы (закон смещения Вина, закон Стефана–Больцмана и спектральный анализ) требуют точных измерений спектров и ручной обработки данных. В то же время в базе обсерватории уже имеются характеристики 240 изученных звёзд: относительная светимость, относительный радиус, абсолютная звёздная величина, цвет, тип звезды и измеренная температура поверхности. С развитием технологий машинного обучения появляется возможность обучить нейросеть на этих данных и предсказывать температуру новых звёзд быстрее, автоматизированно и, возможно, с высокой точностью, сводя к минимуму человеческий фактор при первичной оценке.


### Цель проекта  
Разработать и обучить нейронную сеть для предсказания абсолютной температуры поверхности звезды (T, в Кельвинах) по доступным признакам:
- относительная светимость (L/Lo),  
- относительный радиус (R/Ro),  
- абсолютная звёздная величина (Mv),  
- цвет звезды (категориальный признак),  
- тип звезды (категориальный признак, закодированный числами от 0 до 5).  

Основная задача - добиться качества прогноза (RMSE) не хуже 4500 К и сравнить результаты «базовой» модели нейросети с улучшенной моделью, в которой производится перебор гиперпараметров.


### Описание данных  


1. **L/Lo** (float)  
   - относительная светимость звезды по сравнению с Солнцем.  
   - Единицы измерения: безразмерная величина (L÷L₀), где L₀ = 3.828·10²⁶ Вт.  
   - Признак количественный, может сильно варьироваться (от очень малых значений для коричневых и красных карликов до больших для супергигантов).

2. **R/Ro** (float)  
   - относительный радиус звезды по сравнению с радиусом Солнца.  
   - R₀ = 6.9551·10⁸ м.  
   - Количественный признак, варьируется от долей единицы до сотен (у гигантов и сверхгигантов).

3. **Mv** (float)  
   - абсолютная звёздная величина (величина абсолютного блеска).  
   - Количественный признак: отрицательные значения соответствуют очень ярким звёздам, положительные - тусклым.

4. **Звёздный цвет** (строка)  
   - описание цвета (white, red, blue, yellow, yellow-orange и др.), полученное по спектральному анализу визуального диапазона.  
   - Категориальный признак, требующий перекодировки (one-hot encoding или ordinal encoding).

5. **Тип звезды** (строка)  
   - категория звезды:  
   - «Коричневый карлик», «Красный карлик», «Белый карлик», «Звезда главной последовательности», «Сверхгигант», «Гипергигант».  
   - В датасете дополнительно есть поле с числовым кодом (0–5), соответствующим этим типам.  
   - Категориальный признак, уже численно закодирован.

6. **T(K)** (float)  
   - абсолютная температура поверхности звезды в Кельвинах (целевой признак, y).  
   - Количественный признак, который и требуется предсказать.  

Итого:  
- **Количественные признаки**: L/Lo, R/Ro, Mv, T(K) (T - целевая переменная).  
- **Категориальные признаки**: «Звёздный цвет» (строка), «Тип звезды» (числовой код от 0 до 5).  


### План работы  
1. **Загрузка & EDA**  
   - Импорт данных, быстрое ознакомление (размер, типы, базовые статистики).  
   - Минимальный графический анализ: гистограммы ключевых признаков и распределение целевой температуры.

2. **Предобработка**  
   - Закодировать категориальные (one-hot или целочисленно), масштабировать числовые.  
   - Разбить на обучающую и тестовую выборки.

3. **Baseline-модель**  
   - Определить простую нейросеть (несколько Dense-слоёв, ReLU, выход - линейный).  
   - Обучить, оценить RMSE, построить грубый «Факт-Прогноз».

4. **Улучшенная модель**  
   - Провести ручной перебор ключевых гиперпараметров (dropout, batch size).  
   - Сравнить RMSE, выбрать лучший вариант (RMSE ≤ 4500).

5. **Выводы**  
   - Кратко сравнить baseline и улучшенную модели по метрикам и визуализациям.  
   - Дать рекомендации по дальнейшему улучшению (доп. данные, архитектура и т. д.).

## Загрузка данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from time import time
from math import ceil
import torch
from torch import nn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector, make_column_transformer
from torch.utils.data import Dataset, DataLoader, TensorDataset
pd.options.display.float_format = '{:,.1f}'.format

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

In [ ]:
!pip install -U scikit-learn -q

In [ ]:
try:
    df = pd.read_csv('6_class.csv')
except:
    df = pd.read_csv('https://code.s3.yandex.net/datasets/6_class.csv')
df = df.drop(columns=['Unnamed: 0'])
df.sample(5)

In [ ]:
df.info()

- **Наблюдения:**  
  - Пропусков нет, все 240 строк заполнены полностью во всех столбцах.  
  - Количественные признаки (`Temperature`, `Luminosity`, `Radius`, `Mv`, `Star type`) имеют числовой тип.  
  - `Star color` стоит в формате `object` - придётся преобразовать этот столбец в категориальный (one-hot или аналог).

In [ ]:
df.describe()

**Основные наблюдения и выводы**

1. **Пропуски и «нулевые» значения**  
   - Пропусков нет; однако у ~25% звёзд `Luminosity = 0` и у многих `Radius` близок к нулю. Вероятно, это действительно очень тусклые карлики, но стоит проверить возможность логарифмирования или фильтрации.

2. **Скошенные распределения**  
   - `Luminosity` и `Radius` имеют сильный правый хвост (несколько сверхгигантов «раздувают» среднее), поэтому при масштабе лучше использовать логарифм или RobustScaler.

3. **Целевая переменная (`Temperature`)**  
   - Температуры варьируются от ~2 000 K до 40 000 K (медиана ≈ 5 776 K). Модель должна уметь предсказывать как низкие, так и высокие значения.

4. **Категориальные признаки**  
   - `Star color` - строковый тип с множеством вариантов (разный регистр, дефисы), нужно нормализовать и закодировать (one-hot или аналог).  
   - `Star type` (0–5) распределён относительно равномерно; при разбиении на train/test рекомендуется стратификация по этому столбцу.

In [ ]:
df['Star color'].value_counts()

Имеются дубли и вариации написания (разные регистры, дефисы), что стоит учесть при предобработке категориального признака.

**Итог:**  
- **Данные полные**, но имеются «нулевые» значения в колонках `Luminosity` и `Radius`, которые, скорее всего, отражают очень низкие значения (карлики). При моделировании советуем либо оставить, либо (для более стабильного обучения) применить логарифмирование (например, `log(x + ε)`), либо нормировать при помощи RobustScaler.  
- **Широкий динамический диапазон** у абсолютных величин и физических размеров указывает на необходимость продуманного масштабирования (стандартизация/логарифмирование) перед подачей признаков в нейросеть.  
- **Целевой признак (`Temperature`)** также сильно варьируется, так что модель должна иметь достаточную гибкость, чтобы «поймать» как низкие, так и высокие температуры в разных классах звёзд.  
- **Категориальные признаки** (`Star color`, `Star type`) нужно перекодировать в числовую форму (one-hot или вложенные эмбеддинги), а при разделении на train/test стоит учесть стратификацию по `Star type`.


## Предобработка и анализ данных

Переименуем колонки.

In [ ]:
df = df.rename(columns={
    'Temperature (K)': 'temperature',
    'Luminosity(L/Lo)': 'luminosity',
    'Radius(R/Ro)': 'radius',
    'Absolute magnitude(Mv)': 'absolute_magnitude',
    'Star type': 'star_type',
    'Star color': 'star_color'
})

Нужно проверить неочевидные дубликаты.

In [ ]:
df['star_color'].unique()

Исправим.

In [ ]:
df['star_color'] = (
    df['star_color']
    .str.lower()
    .str.replace('-', ' ', regex=False)
    .str.replace('ish', '', regex=False)
    .str.strip()
)

def renamer(x):
    if x == 'white yellow':
        return 'yellow white'
    elif x == 'whit':
        return 'white'
    else:
        return x
    
df['star_color'] = df['star_color'].apply(renamer)
df['star_color'].unique()    

Мы можем перевести категорию «цвет звезды» в вещественный признак, близкий к реальной физической температуре, и одновременно логарифмически сгладить диапазон, чтобы модель могла корректнее учитывать этот фактор при обучении.

In [ ]:

color_to_temp = {
    'blue': 30000,
    'blue white': 20000,
    'white': 9000,
    'yellow white': 7000,
    'yellow': 5500,
    'pale yellow orange': 5000,
    'orange': 4500,
    'orange red': 3750,
    'red': 3000
}


df['avg_temp'] = np.log(df['star_color'].map(color_to_temp))

# теперь у каждой звезды в df будет столбец avg_temp с приближённой температурой

Определим признаки и таргет.

In [ ]:
TARGET = ["temperature"]
numerical_features   = ["luminosity", "radius", "absolute_magnitude", "avg_temp"]
categorical_features = ["star_type", "star_color"]

Далее рассмотрим распределение каждого признака.

In [ ]:
for feature in numerical_features:
    
    fig, (ax1, ax2) = plt.subplots(2, sharex=True, gridspec_kw={"height_ratios": (.8, .2)}, figsize=(12, 8))
    plt.title(f"Распределение признака «{feature}»", fontsize=14)
    sns.boxplot(x=df[feature], ax=ax2, color='royalblue')
    sns.kdeplot(df[feature],ax=ax1, fill=True)    
        
    plt.title(f"Распределение: {feature}", fontsize=14)
    plt.xlabel(feature, fontsize=12)
    ax1.set_ylabel("Плотность", fontsize=12)
    plt.tight_layout()
    plt.show()
    
fig, (ax1, ax2) = plt.subplots(2, sharex=True, gridspec_kw={"height_ratios": (.8, .2)}, figsize=(12, 8))
plt.title(f"Распределение температуры", fontsize=14)
sns.boxplot(x=df['temperature'], ax=ax2, color='royalblue')
sns.kdeplot(df['temperature'],ax=ax1, fill=True)    

plt.title(f"Распределение температуры", fontsize=14)
plt.xlabel('Температура', fontsize=12)
ax1.set_ylabel("Плотность", fontsize=12)
plt.tight_layout()
plt.show()


### Интерпретация распределений

#### 1. Luminosity (L/Lo)
- **Форма распределения:** сильно скошено вправо (KDE «тяжёлый» правый хвост).  
- **Большинство значений:** сосредоточено ближе к нижней границе (0 … ~200 000 L₀), при этом 25 % звёзд имеют L≈0.  
- **Выбросы:** несколько объектов со светимостью до ~800 000 L₀ (сверхгиганты) заметно «разгоняют» правый хвост.  

#### 2. Radius (R/Ro)
- **Форма распределения:** опять же сильный правый хвост.  
- **Большинство значений:** радиусы до ~100 R₀ (медиана ~0.8 R₀, 75 % ≤ ~42 R₀).  
- **Выбросы:** отдельные объекты с радиусом до ~2 000 R₀ (гиганты и сверхгиганты).  
- **Вывод:** аналогично светимости, лучше использовать RobustScaler, чтобы «смягчить» влияние крупных радиусов.

#### 3. Absolute magnitude (Mv)
- **Форма распределения:** бимодальное распределение.  
  - Один пик на отрицательных Mv (~–10 … 0): очень яркие (сверхгиганты).  
  - Второй пик на положительных Mv (~5 … 15): тусклые карлики и главная последовательность.  
- **Диапазон:** от –11.9 до +20; медиана ≈ 8.3.  
- **Вывод:** наличие двух явных групп отражает разные типы звёзд (яркие vs. тусклые); модель должна уметь «различать» эти кластеры.

#### 4. avg_temp (логарифм приближённой температуры по цвету)
- **Форма распределения:** тоже бимодальное.  
  - Один пик около log(5500 K… 7000 K) ≈ 8.6 … 8.9 (жёлтые звёзды, главная последовательность).  
  - Второй пик около log(20000 K… 30000 K) ≈ 10.0 … 10.3 (голубые и голубо-белые звёзды).  
- **Диапазон:** приблизительно от ~7 (очень холодные: red) до ~11.2 (очень горячие: blue).  
- **Вывод:** по цвету выделяются две группы («теплые» vs. «холодные»).

#### 5. Temperature (целевая переменная)
- **Форма распределения:** сильный правый хвост.  
- **Большинство значений:** лежит в диапазоне ~2000 … 15 000 K (медиана ~5776 K, как у Солнца).  
- **Выбросы:** несколько «горячих» звёзд до ~40 000 K.  
- **Вывод:** целевая температура охватывает широкий диапазон, модель должна уметь точно предсказывать как низкие (~2000 K), так и высокие (~40 000 K) значения.

---




In [ ]:
for feature in categorical_features:
    plt.figure(figsize=(12, 10))
    sns.countplot(x=feature, data=df, order = df[feature].value_counts().index, color="skyblue")
    plt.title(f"Распределение: {feature}", fontsize=14)
    plt.xlabel(feature, fontsize=12)
    plt.ylabel("Плотность", fontsize=12)
    if feature == 'star_color':
        plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


### Распределение категориальных признаков

#### 1. Распределение `star_type`
- Всего 6 категорий с кодами от 0 до 5 (соответствуют: коричневый карлик, красный карлик, белый карлик, звезда главной последовательности, сверхгигант и гипергигант).
- На гистограмме видно, что у каждой из шести категорий ровно по 40 объектов (240 / 6 = 40).  
- **Вывод:** сбалансированная выборка по типам звёзд.

---

#### 2. Распределение `star_color`
- Категорий больше (примерно 9 основных), но распределение очень неравномерное:
  - **red** – 112 объектов (≈ 47 % от всей выборки),
  - **blue** – 56 объектов (≈ 23 %),
  - **blue white** – 41 объект (≈ 17 %),
  - Затем «yellow white» и «white» – по 12 каждый (≈ 5 %),
  - И совсем редкие «yellow», «orange», «pale yellow orange», «orange red» (в сумме ≈ 7 %).
- **Вывод:**  
  1. Чисто по цвету наблюдается сильный перекос в сторону «red» и «blue», остальные «теплые» и «белые» классы встречаются гораздо реже.  
  2. Поскольку мы заменяем `star_color` на численный признак `avg_temp` (логарифм средней температуры по цвету), исходный столбец `star_color` будет удалён. Это позволяет избавиться от дисбаланса и «шума» множественных строковых категорий, сохранив физически значимую информацию о цвете через `avg_temp`.


In [ ]:
if 'star_color' in categorical_features:
    categorical_features.remove('star_color')

## Преобразование и подготовка данных

Разбиваем на test/train подвыборки. Так как выборка очень маленькая возьмем 20% для валидации.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=['temperature']),
    df.temperature,
    test_size=0.2,
    shuffle=True,
    random_state=10
)


X_train.shape, X_test.shape, y_train.shape, y_test.shape

Определяем пайплайн и как мы будем обрабатывать признаки. Чусленные - StandardScaler. Категориальные - OneHotEncoding.

In [ ]:
numerical_processor = Pipeline([("scaler", StandardScaler())])
categorical_processor = Pipeline([("onehot", OneHotEncoder(sparse_output=False))])

Preprocess = ColumnTransformer(
    transformers=[
        ("num", numerical_processor, numerical_features),
        ("cat", categorical_processor, categorical_features),
    ],
    remainder="drop",  
)


In [ ]:
import sklearn
sklearn.__version__

In [ ]:
X_train = Preprocess.fit_transform(X_train)
X_test = Preprocess.transform(X_test)


In [ ]:
X_train = torch.FloatTensor(np.array(X_train))
y_train = torch.FloatTensor(np.array(y_train))
X_test = torch.FloatTensor(np.array(X_test))
y_test = torch.FloatTensor(np.array(y_test))

## Построение базовой нейронной сети

### Baseline

**Архитектура сети `Net`:**
- Входной слой: `Linear(n_in → n_hidden_1)`
  - BatchNorm1d(n_hidden_1)
  - ReLU
  - Dropout(p_dropout)
- Скрытый слой 2: `Linear(n_hidden_1 → n_hidden_2)`
  - BatchNorm1d(n_hidden_2)
  - ReLU
- Выходной слой: `Linear(n_hidden_2 → n_out)`

**Инициализация весов:**
- Для всех `Linear`-слоёв:
  - Веса - Kaiming uniform (He-инициализация) под ReLU
  - Сдвиги (bias) - нормальное распределение (mean=0.5, std=0.7)

In [ ]:
class Net(nn.Module):
    """
    Простая модель с двумя скрытыми слоями, BatchNorm, ReLU.
    """
    def __init__(
        self,
        n_in: int,
        n_hidden_1: int,
        n_hidden_2: int,
        n_out: int,
        p_dropout: float = 0.0
    ) -> None:
        super(Net, self).__init__()

        # Определяем последовательность слоёв
        self.net = nn.Sequential(
            nn.Linear(n_in, n_hidden_1),
            nn.BatchNorm1d(n_hidden_1),
            nn.ReLU(),
            nn.Dropout(p=p_dropout),
            nn.Linear(n_hidden_1, n_hidden_2),
            nn.BatchNorm1d(n_hidden_2),
            nn.ReLU(),
            nn.Linear(n_hidden_2, n_out)
        )

        # Инициализация весов 
        for module in self.net:
            if isinstance(module, nn.Linear):
                nn.init.kaiming_uniform_(module.weight, mode='fan_in', nonlinearity='relu')
                nn.init.normal_(module.bias, mean=.5, std=.7)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

- Мы строим MLP с двумя скрытыми слоями: 32 нейрона → 16 нейронов → 1 нейрон на выходе.

- Обучаем её оптимизатором Adam с lr=1e-3 и лёгкой L2-регуляризацией (weight_decay=1e-5).

- В качестве функции потерь берём MSE, поскольку решаем задачу регрессии (прогноз температуры).

In [ ]:
def net_init(
    n_in: int,
    n_hidden_1: int,
    n_hidden_2: int,
    n_out: int,
    p_dropout: float = 0.0,
    lr: float = 1e-3,
    seed: int = 42
):
    """
    Инициализирует модель, оптимизатор Adam и MSELoss, задавая сид для воспроизводимости.

    Args:
        n_in (int): размер входного вектора.
        n_hidden_1 (int): число нейронов в первом скрытом слое.
        n_hidden_2 (int): число нейронов во втором скрытом слое.
        n_out (int): размер выходного вектора.
        p_dropout (float): вероятность отключения нейронов (Dropout).
        lr (float): learning rate для оптимизатора.
        seed (int): сид для torch.manual_seed.

    Returns:
        model (Net): инициализированная модель.
        optimizer (torch.optim.Optimizer): оптимизатор Adam.
        loss_fn (nn.MSELoss): функция потерь (MSE).
    """
    torch.manual_seed(seed)
    model = Net(n_in, n_hidden_1, n_hidden_2, n_out, p_dropout).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.MSELoss()
    return model, optimizer, loss_fn



n_in = X_train.shape[1]
n_hidden_1 = 32
n_hidden_2 = 16
n_out = 1
p_dropout = 0.0

model, optimizer, loss_fn = net_init(
    n_in=n_in,
    n_hidden_1=n_hidden_1,
    n_hidden_2=n_hidden_2,
    n_out=n_out,
    p_dropout=p_dropout,
    lr=1e-3,
    seed=42
)


Как только валидационная потеря не демонстрирует улучшения (на величину ≥ `min_delta`) в течение `patience` подряд эпох, тренировка прекращается заранее, даже если заранее задано большее число эпох.

In [ ]:
class EarlyStop:
    """
    Контролирует раннюю остановку обучения, если валидационные потери не улучшаются.
    """
    def __init__(self, patience: int = 5, min_delta: float = 1e-3) -> None:
        """
        Args:
            patience (int): сколько подряд проверок ждать улучшения.
            min_delta (float): минимальное изменение, считаемое улучшением (в абсолютном значении).
        """
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, current_loss: torch.Tensor) -> None:
        """
        Вызывается при каждой проверке валидационных потерь.

        Args:
            current_loss (torch.Tensor): текущее значение RMSE на валидационном наборе.
        """
        loss_value = current_loss.item()
        # Проверяем, есть ли улучшение
        if self.best_loss - loss_value >= self.min_delta:
            self.best_loss = loss_value
            self.counter = 0
        else:
            self.counter += 1
            if self.counter > self.patience:
                print('--- Early stopping triggered ---')
                self.early_stop = True


Функция обучения.

In [ ]:
def net_training(
    model: Net,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.MSELoss,
    X_train: torch.Tensor,
    y_train: torch.Tensor,
    X_val: torch.Tensor,
    y_val: torch.Tensor,
    n_epoch: int = 100_000,
    batch_size: int = 40,
    stop_interval: int = 100,
    stop_patience: int = 5,
    stop_delta: float = 1e-3,
    verbose: int = 0
):
    """
    Обучает модель, проверяет валид. потери каждые stop_interval эпох, 
    и прекращает обучение, если улучшения нет в течение stop_patience проверок.

    Args:
        model (Net): модель для обучения.
        optimizer (torch.optim.Optimizer): оптимизатор.
        loss_fn (nn.MSELoss): функция потерь.
        X_train (torch.Tensor): тензор признаков для тренировки.
        y_train (torch.Tensor): тензор таргетов для тренировки.
        n_epoch (int): максимальное число эпох.
        batch_size (int): размер батча.
        stop_interval (int): через сколько эпох проверять валидацию.
        stop_patience (int): число подряд проверок без улучшения.
        stop_delta (float): минимальное изменение RMSE, считающееся улучшением.
        verbose (int): если > 0, выводим прогресс каждые stop_interval * verbose эпох.

    Returns:
        best_pred (torch.Tensor | None): предсказание модели при лучшей RMSE (или None, если не было проверок).
        best_loss (float): лучшее зафиксированное значение RMSE на валидации.
    """
    # Переносим все данные на указанное устройство
    X_train, y_train = X_train.to(DEVICE), y_train.to(DEVICE)
    X_val, y_val = X_val.to(DEVICE), y_val.to(DEVICE)

    # Объект ранней остановки
    early_stop = EarlyStop(patience=stop_patience, min_delta=stop_delta)

    best_loss = float('inf')
    best_pred = None

    for epoch in range(1, n_epoch + 1):
        # Генерируем новый DataLoader каждый раз, чтобы перетасовывать данные
        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0
        )

        model.train()
        for batch_features, batch_targets in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_features)

            loss_train = loss_fn(outputs.flatten(), batch_targets.flatten())
            loss_train.backward()
            optimizer.step()

        # Проверка валидации каждые stop_interval эпох
        if epoch % stop_interval == 0:
            model.eval()
            with torch.no_grad():
                val_outputs = model(X_val)
                loss_valid = loss_fn(val_outputs.flatten(), y_val.flatten())

            if verbose > 0 and (epoch % (stop_interval * verbose) == 0):
                print(f'Epoch {epoch:6d} | RMSE_valid = {torch.sqrt(loss_valid):.1f}')

            # Проверяем EarlyStop
            early_stop(loss_valid)
            # Если улучшение, сохраняем результаты
            if early_stop.counter == 0:
                best_loss = loss_valid.item()
                best_pred = val_outputs.detach().cpu()
            # Случай раннего останова
            if early_stop.early_stop:
                print(f'>>> Training stopped. Best RMSE_valid = {np.sqrt(best_loss):.1f}')
                return best_pred, np.sqrt(best_loss)

    # Если дошли до конца всех эпох
    return best_pred, np.sqrt(best_loss)

Можно начать обучать!

In [ ]:
%%time

prediction, best_rmse = net_training(
    model=model,
    optimizer=optimizer,
    loss_fn=loss_fn,
    X_train=X_train,
    y_train=y_train,
    X_val=X_test,
    y_val=y_test,
    n_epoch=100_000,
    batch_size=60,
    stop_interval=100,
    stop_patience=20,
    stop_delta=1e-2,
    verbose=20
)

print(f'Итоговая лучшая RMSE на валидации: {best_rmse:.1f}')

Модель законцила обичение с RMSE: 2899. Хороший результат, можно попробовать улучшить.

Визуализируем результаты.

In [ ]:
baseline_model_prediction = prediction
baseline_model_rmse = best_rmse

In [ ]:
actual    = y_test.detach().numpy().reshape([-1])
predicted = baseline_model_prediction.detach().numpy().reshape([-1])


# Количество звёзд
N = len(actual)


indices = np.arange(N)

bar_width = 0.35

# Создаём фигуру нужного размера
plt.figure(figsize=(16, 8))


plt.bar(indices - bar_width/2, 
        actual, 
        width=bar_width, 
        color='#87CEEB',  
        label='Факт')


plt.bar(indices + bar_width/2, 
        predicted, 
        width=bar_width, 
        color='#FFD700',  
        label='Прогноз')

plt.xlabel('Номер звезды в таблице данных', fontsize=12)
plt.ylabel('Температура звезды', fontsize=12)
plt.title(f'Фактическая и прогнозная температура звезд {baseline_model_rmse:.0f}', fontsize=14)


plt.xticks(indices, [str(i) for i in indices])

# Добавляем сетку по горизонтали (опционально)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Легенда
plt.legend()

# Показать
plt.tight_layout()
plt.show()

На графике показано сравнение фактической температуры поверхности звёзд (голубые столбцы) и температур, предсказанных моделью (золотые столбцы) для выборки объектов. В большинстве случаев высота золотых столбцов (прогноз) близка к высоте голубых (реальные значения), хотя встречаются отдельные точки, где модель заметно завышает или занижает значение. В целом этот график наглядно показывает, насколько хорошо регрессионная модель повторяет реальные измерения температуры для первых 48 звёзд выборки.  


### Улучшение нейронной сети

Чтобы улучшить нашу модель рассмотрим как влияет p_dropout и размер батча на нашу метрику.

In [ ]:
%%time
 
scores = pd.DataFrame(columns=['batch_size', 'p_dropout', 'rmse', 'prediction'])

# гиперпараметры для перебора
batch_size_list = [16, 32, 64]
p_dropout_list = [0.1, 0.2 , 0.3, 0.4 ,0.5, 0.6, 0.7, 0.8, 0.9]# , 0.7, 0.8 ] # 

for batch_size in batch_size_list:
    for p_dropout in p_dropout_list:
    
        start_time = time()
        print(f'--- batch_size={batch_size}, p_dropout={p_dropout} ---')

        # инициализация очередной модели
        model, optimizer, loss = net_init(n_in, n_hidden_1, n_hidden_2, n_out, p_dropout=p_dropout)        

        # обучение модели
        prediction, best_rmse = net_training(model=model,
                                        optimizer=optimizer,
                                        loss_fn=loss_fn,
                                        X_train=X_train,
                                        y_train=y_train,
                                        X_val=X_test,
                                        y_val=y_test,
                                        n_epoch=100000, 
                                        batch_size=batch_size,
                                        stop_interval=100, 
                                        stop_patience=20, 
                                        stop_delta=1e-3, 
                                        verbose=5)

        print(f'time spent: {time() - start_time:.1f} sec.')

        # добавление очередного результата в таблицу
        scores.loc[len(scores)] = [batch_size, p_dropout, best_rmse, prediction]

Таблица результатов

In [ ]:
scores = scores.sort_values('rmse').reset_index(drop=True)
scores

Визуализируем.

In [ ]:
plt.figure(figsize=(12, 10)) 
sns.lineplot(
    data=scores,
    x='p_dropout',
    y='rmse',
    hue='batch_size',
    style='batch_size',
    markers=True,
    dashes=False
)

plt.axhline(
    y=baseline_model_rmse,
    color='red',       # line color
    linestyle='--',    # dashed style
    linewidth=1.5,     # thickness
    label=f'Baseline RMSE: {baseline_model_rmse:.0f}'
)

plt.xlabel('p_dropout')
plt.ylabel('RMSE')
plt.title('RMSE vs p_dropout для разных батчей (Batch Size)')
plt.legend(title='Размер батча')
plt.show()

**Выводы:**  
- Лучшие комбинации (чуть лучше чем baseline = 2899) получены при очень малом дропауте:
  - **batch_size=32, p_dropout=0.3** (2,860.2 RMSE),
  - **batch_size=64, p_dropout=0.1** (2,863.4 RMSE).  
- Как только `p_dropout` переваливает за ~0.3–0.4, RMSE начинает расти, и к p≈0.7–0.9 качество падает критически.  
- Значит, для данной архитектуры оптимально держать дропаут примерно **0.1–0.2**, а размер батча можно выбирать из {32, 64} (32 чуть лучше на этих данных).  
- Все остальные комбинации дают RMSE выше baseline, поэтому они считаются менее удачными.

Визуализируем результаты лучшей модели.

In [ ]:
best_model_predictions = scores.loc[0, 'prediction']
actual    = y_test.detach().numpy().reshape([-1])
predicted = best_model_predictions.detach().numpy().reshape([-1])


# Количество звёзд
N = len(actual)


indices = np.arange(N)

# Ширина каждого бара (столбика). Обычно 0.35–0.4 даёт аккуратный вид
bar_width = 0.35

# Создаём фигуру нужного размера
plt.figure(figsize=(16, 8))

# Рисуем первый набор столбиков (Факт), сдвинутый влево на bar_width/2
plt.bar(indices - bar_width/2, 
        actual, 
        width=bar_width, 
        color='#87CEEB',  
        label='Факт')

# Рисуем второй набор столбиков (Прогноз), сдвинутый вправо на bar_width/2
plt.bar(indices + bar_width/2, 
        predicted, 
        width=bar_width, 
        color='#FFD700',  
        label='Прогноз')

# Подписи осей и заголовок
plt.xlabel('Номер звезды в таблице данных', fontsize=12)
plt.ylabel('Температура звезды', fontsize=12)
plt.title(f"Фактическая и прогнозная температура звезд. RMSE: {scores.loc[0, 'rmse']:.0f}, p_dropout: {scores.loc[0, 'p_dropout']}, batch_size: {scores.loc[0, 'batch_size']}", fontsize=14)

# Настройка «хитростей» для отображения чисел по X
# Вместо 0.00, 1.00, ... покажем просто целые индексы: 0, 1, 2, ...
plt.xticks(indices, [str(i) for i in indices])

# Добавляем сетку по горизонтали (опционально)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Легенда
plt.legend()

# Показать
plt.tight_layout()
plt.show()

## Сравнение моделей

| Модель                | `batch_size` | `p_dropout` | RMSE (тест) | Улучшение относительно baseline |
|-----------------------|--------------|-------------|-------------|---------------------------------|
| **Baseline**          | 60           | 0.0         | 2899        | -                               |
| **Улучшенная модель** | **32**       | **0.3**     | **2860**    | ~39 пунктов (≈1.3 %)            |
| Альтернатива          | 64           | 0.1         | 2863        | ~36 пунктов (≈1.2 %)            |


## Резюме проекта и итоги.

1. EDA и предобработка
    - Данные: 240 строк.
    - Пропусков нет; у четверти звёзд `Luminosity=0` и `Radius≈0` (карлики), есть сверхгиганты с огромными значениями. Распределения сильно скошены вправо.
    - `Absolute magnitude` и `avg_temp` (лог T по цвету) имеют бимодальные распределения.
    - `Star color` заменили на численный признак `avg_temp = log(T_approx по цвету)` и отбросили исходный.

2. Baseline-модель
    - Архитектура:  `[n_in → 32] → BatchNorm → ReLU → Dropout(0) → [32 → 16] → BatchNorm → ReLU → [16 → 1]`.
    - Инициализация: веса - Kaiming uniform, bias - N(0.5, 0.7).
    - Оптимизатор: Adam(lr=1e-3, weight_decay=1e-5), loss - MSE.
    - Ранний стоп: patience=20, min_delta=1e-3.
    - Результат baseline: **RMSE ≈ 2899** на тесте.

3. Перебор `batch_size` и `p_dropout`
    - Перебрали `batch_size ∈ {16, 32, 64}`, `p_dropout ∈ {0.1…0.9}` (lr=1e-3, остальные условия те же).
    - Лучшее:
    - `batch_size=32`, `p_dropout=0.3` → **RMSE ≈ 2860** (≈1.3 % лучше baseline).
    - Альтернатива: `batch_size=64`, `p_dropout=0.1` → RMSE ≈ 2863.
    - Сильный дропаут (p ≥ 0.3) резко ухудшает модель (RMSE ≥ 3000→4500).

4. Главные выводы
    - **Оптимальная конфигурация**: MLP (32→16→1) + `bs=32`, `p_dropout=0.2` → RMSE ≈ 2860.
    - **Рекомендации**:
        1. Протестировать другие функции активации (LeakyReLU, ELU) и optimizers (RAdam).
        2. При росте выборки (>240) можно увеличить число скрытых слоёв или добавить остаточные связи.
